# 🩺 Diabetes Prediction — Complete Machine Learning Pipeline

This notebook follows the same overall structure as the Heart Disease notebook you provided, but is adapted to the uploaded diabetes dataset.

## Pipeline
1. Import libraries
2. Load and inspect the dataset
3. Data validation and duplicate checking
4. Detect medically implausible zero values used as missing-value placeholders
5. Descriptive statistics
6. Target distribution
7. Missing-value analysis
8. Histograms / distributions
9. Boxplots and outlier inspection
10. Feature vs. diabetes-outcome analysis
11. Correlation analysis
12. Pairwise relationships
13. Train/test split
14. Leakage-safe preprocessing
15. Train 6 models
16. Test-set evaluation
17. Confusion matrices
18. Metric comparison graphs
19. ROC curves
20. Precision–Recall curves
21. Cross-validation
22. Random Forest feature importance
23. Logistic Regression coefficients
24. Linear Regression residual analysis
25. Final model comparison and saving outputs

> **Important:** `Outcome` is the target: `0 = No Diabetes`, `1 = Diabetes`.


In [ ]:
# STEP 1 — Import libraries
import warnings
warnings.filterwarnings("ignore")
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score,
    roc_curve, precision_recall_curve
)

RANDOM_STATE = 42


In [ ]:
# STEP 2 — Load the dataset
# The first name matches the GitHub repo structure; the second matches the uploaded file.
if os.path.exists("diabetes.csv"):
    DATA_PATH = "diabetes.csv"
elif os.path.exists("diabetes (1).csv"):
    DATA_PATH = "diabetes (1).csv"
else:
    raise FileNotFoundError(
        "Put diabetes.csv (or diabetes (1).csv) in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Loaded file:", DATA_PATH)
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget values:")
print(sorted(df["Outcome"].unique()))


In [ ]:
# STEP 3 — Validate the target and check data quality
assert set(df["Outcome"].unique()).issubset({0, 1}), "Outcome must contain only 0 and 1."

print("Rows before cleaning:", len(df))
print("Exact duplicate rows:", df.duplicated().sum())

df_clean = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates:", len(df_clean))
print("Duplicates after cleaning:", df_clean.duplicated().sum())
print("Total explicit missing values:", df_clean.isna().sum().sum())


## STEP 4 — Zero-value analysis

In this commonly used diabetes dataset, zero is not physiologically meaningful for several measurements. We therefore inspect zero counts before modeling.

We treat zeros as missing only for:
- `Glucose`
- `BloodPressure`
- `SkinThickness`
- `Insulin`
- `BMI`

`Pregnancies = 0` is valid, and `Age = 0` is not present in normal records, so we do not blindly replace every zero in the dataset.


In [ ]:
zero_check_cols = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
]

zero_counts = (df_clean[zero_check_cols] == 0).sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
plt.bar(zero_counts.index, zero_counts.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of zero values")
plt.title("Zero-Value Analysis by Feature")
plt.tight_layout()
plt.show()

display(zero_counts.to_frame("zero_count"))


In [ ]:
# STEP 5 — Replace placeholder zeros with NaN
zero_as_missing = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

df_model = df_clean.copy()
df_model[zero_as_missing] = df_model[zero_as_missing].replace(0, np.nan)

print("Missing values after zero-value treatment:")
display(df_model.isna().sum().sort_values(ascending=False))

print("Total missing values after treatment:",
      int(df_model.isna().sum().sum()))


In [ ]:
# STEP 6 — Target distribution
target_counts = df_model["Outcome"].value_counts().sort_index()

plt.figure(figsize=(7, 5))
plt.bar(["0 = No Diabetes", "1 = Diabetes"], target_counts.values)
plt.ylabel("Number of patients")
plt.title("Diabetes Outcome Distribution")

for i, value in enumerate(target_counts.values):
    plt.text(i, value + max(target_counts.values)*0.01, str(value),
             ha="center")

plt.tight_layout()
plt.show()

print("Target counts:")
display(target_counts)

print("Target percentages:")
display((df_model["Outcome"].value_counts(normalize=True).sort_index() * 100).round(2))


In [ ]:
# STEP 7 — Descriptive statistics
display(df_model.describe().T)


In [ ]:
# STEP 8 — Missing-value percentage
missing_pct = (
    df_model.isna().mean()
    .mul(100)
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
plt.bar(missing_pct.index, missing_pct.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Missing values (%)")
plt.title("Missing-Value Percentage by Feature")
plt.tight_layout()
plt.show()

display(missing_pct.to_frame("missing_percentage"))


# 📊 Exploratory Data Analysis

The following graphs are designed to answer:
- What does each feature look like?
- Which features contain extreme values?
- How do feature distributions differ between diabetic and non-diabetic patients?
- Which variables appear related to the target?
- Which features are correlated with one another?


In [ ]:
# STEP 9 — Histograms for all numerical features
feature_cols = [c for c in df_model.columns if c != "Outcome"]

for col in feature_cols:
    plt.figure(figsize=(8, 5))
    plt.hist(df_model[col].dropna(), bins=25)
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.show()


In [ ]:
# STEP 10 — Boxplots for all numerical features
for col in feature_cols:
    plt.figure(figsize=(8, 5))
    plt.boxplot(df_model[col].dropna(), vert=True)
    plt.ylabel(col)
    plt.title(f"Boxplot of {col}")
    plt.tight_layout()
    plt.show()


In [ ]:
# STEP 11 — Feature vs Diabetes: boxplots
for col in feature_cols:
    group0 = df_model.loc[df_model["Outcome"] == 0, col].dropna()
    group1 = df_model.loc[df_model["Outcome"] == 1, col].dropna()

    plt.figure(figsize=(8, 5))
    plt.boxplot(
        [group0, group1],
        labels=["No Diabetes (0)", "Diabetes (1)"]
    )
    plt.ylabel(col)
    plt.title(f"{col} vs Diabetes Outcome")
    plt.tight_layout()
    plt.show()


In [ ]:
# STEP 12 — Feature vs Diabetes: mean comparison
mean_by_target = df_model.groupby("Outcome")[feature_cols].mean().T

display(mean_by_target)

for col in feature_cols:
    means = df_model.groupby("Outcome")[col].mean()

    plt.figure(figsize=(7, 5))
    plt.bar(["No Diabetes", "Diabetes"], means.values)
    plt.ylabel(f"Mean {col}")
    plt.title(f"Mean {col} by Diabetes Outcome")

    for i, v in enumerate(means.values):
        plt.text(i, v, f"{v:.2f}", ha="center", va="bottom")

    plt.tight_layout()
    plt.show()


In [ ]:
# STEP 13 — Binned feature vs Diabetes: disease rate
for col in feature_cols:
    temp = df_model[[col, "Outcome"]].dropna().copy()

    # Use quantile bins where possible
    try:
        temp["bin"] = pd.qcut(temp[col], q=5, duplicates="drop")
    except ValueError:
        continue

    grouped = temp.groupby("bin", observed=True)["Outcome"].agg(
        ["mean", "count"]
    ).reset_index()

    plt.figure(figsize=(9, 5))
    x = np.arange(len(grouped))
    plt.bar(x, grouped["mean"])
    plt.xticks(x, grouped["bin"].astype(str), rotation=35, ha="right")
    plt.ylim(0, 1)
    plt.ylabel("Diabetes rate")
    plt.xlabel(f"{col} quantile")
    plt.title(f"{col} vs Diabetes — Diabetes Rate by Range")

    for i, v in enumerate(grouped["mean"]):
        plt.text(i, min(v + 0.02, 0.98), f"{v:.1%}", ha="center")

    plt.tight_layout()
    plt.show()


In [ ]:
# STEP 14 — Correlation matrix
corr = df_model.corr(numeric_only=True)

plt.figure(figsize=(11, 9))
plt.imshow(corr.values, aspect="auto")
plt.colorbar(label="Correlation")

plt.xticks(
    range(len(corr.columns)),
    corr.columns,
    rotation=90
)
plt.yticks(
    range(len(corr.columns)),
    corr.columns
)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

print("Correlation with Outcome:")
display(
    corr["Outcome"]
    .sort_values(ascending=False)
    .to_frame("correlation_with_outcome")
)


In [ ]:
# STEP 15 — Pairwise scatter plots against Outcome
# A compact pairwise view for every feature vs Outcome.
for col in feature_cols:
    plt.figure(figsize=(8, 5))

    for outcome_value in [0, 1]:
        subset = df_model[df_model["Outcome"] == outcome_value]
        plt.scatter(
            subset[col],
            subset["Outcome"],
            alpha=0.35,
            label="No Diabetes" if outcome_value == 0 else "Diabetes"
        )

    plt.xlabel(col)
    plt.ylabel("Outcome")
    plt.title(f"{col} vs Diabetes Outcome")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# STEP 16 — Pairwise feature relationship matrix using pandas
# This can be computationally heavier on larger datasets.
pd.plotting.scatter_matrix(
    df_model[feature_cols + ["Outcome"]],
    figsize=(16, 16),
    diagonal="hist",
    alpha=0.25
)
plt.suptitle("Pairwise Feature Relationships", y=0.90)
plt.show()


# 🤖 Machine Learning

We compare the same six model families used in the reference Heart Disease notebook:

1. Linear Regression
2. Logistic Regression
3. Decision Tree
4. Random Forest
5. Support Vector Machine (SVM)
6. K-Nearest Neighbors (KNN)

For the five classification models, the primary metrics are Accuracy, Precision, Recall, F1, ROC-AUC and PR-AUC.

Linear Regression is included because it was part of the reference pipeline. Since this is a binary classification problem, its continuous output is thresholded at 0.5 for classification metrics, and MAE/RMSE/R² are also reported.


In [ ]:
# STEP 17 — Separate features and target
X = df_model.drop(columns="Outcome")
y = df_model["Outcome"]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# STEP 18 — Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).sort_index())

print("\nTesting target distribution:")
display(y_test.value_counts(normalize=True).sort_index())


In [ ]:
# STEP 19 — Preprocessing pipeline
# Median imputation handles missing values.
# StandardScaler is important for Logistic Regression, SVM and KNN.
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [ ]:
# STEP 20 — Define all models
models = {
    "Linear Regression": LinearRegression(),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "SVM": SVC(
        probability=True,
        random_state=RANDOM_STATE
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=7
    )
}


In [ ]:
# STEP 21 — Train every model and calculate evaluation metrics
results = []
trained_models = {}
predictions = {}
scores = {}

for name, estimator in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    pipe.fit(X_train, y_train)

    if name == "Linear Regression":
        y_score = pipe.predict(X_test)
        y_pred = (y_score >= 0.5).astype(int)

        mae = mean_absolute_error(y_test, y_score)
        rmse = np.sqrt(mean_squared_error(y_test, y_score))
        r2 = r2_score(y_test, y_score)

    else:
        y_pred = pipe.predict(X_test)
        y_score = pipe.predict_proba(X_test)[:, 1]

        mae = np.nan
        rmse = np.nan
        r2 = np.nan

    trained_models[name] = pipe
    predictions[name] = y_pred
    scores[name] = y_score

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_score),
        "PR_AUC": average_precision_score(y_test, y_score),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

display(results_df.round(4))


In [ ]:
# STEP 22 — Detailed classification reports
for name in models:
    print("\n" + "=" * 75)
    print(name)
    print("=" * 75)

    print(
        classification_report(
            y_test,
            predictions[name],
            target_names=["No Diabetes", "Diabetes"],
            zero_division=0
        )
    )


In [ ]:
# STEP 23 — Confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for ax, name in zip(axes, models):
    cm = confusion_matrix(y_test, predictions[name])

    ax.imshow(cm)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["No Diabetes", "Diabetes"])
    ax.set_yticklabels(["No Diabetes", "Diabetes"])

    for i in range(2):
        for j in range(2):
            ax.text(
                j, i,
                cm[i, j],
                ha="center",
                va="center",
                fontsize=13
            )

plt.tight_layout()
plt.show()


In [ ]:
# STEP 24 — Metric comparison: Accuracy
plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["Accuracy"])
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Model Comparison — Accuracy")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# STEP 25 — Metric comparison: Precision
plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["Precision"])
plt.ylim(0, 1)
plt.ylabel("Precision")
plt.title("Model Comparison — Precision")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# STEP 26 — Metric comparison: Recall
plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["Recall"])
plt.ylim(0, 1)
plt.ylabel("Recall")
plt.title("Model Comparison — Recall")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# STEP 27 — Metric comparison: F1 Score
plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["F1"])
plt.ylim(0, 1)
plt.ylabel("F1 Score")
plt.title("Model Comparison — F1 Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# STEP 28 — Metric comparison: ROC-AUC
plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["ROC_AUC"])
plt.ylim(0, 1)
plt.ylabel("ROC-AUC")
plt.title("Model Comparison — ROC-AUC")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# STEP 29 — Metric comparison: PR-AUC
plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["PR_AUC"])
plt.ylim(0, 1)
plt.ylabel("PR-AUC")
plt.title("Model Comparison — Precision-Recall AUC")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# STEP 30 — All classification metrics in one figure
metric_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, metric in zip(axes, metric_cols):
    ax.barh(results_df["Model"], results_df[metric])
    ax.set_xlim(0, 1)
    ax.set_title(metric)
    ax.set_xlabel(metric)

plt.tight_layout()
plt.show()


In [ ]:
# STEP 31 — ROC curves for all models
plt.figure(figsize=(9, 7))

for name in models:
    fpr, tpr, _ = roc_curve(y_test, scores[name])
    auc = roc_auc_score(y_test, scores[name])
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Model Comparison")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# STEP 32 — Precision-Recall curves for all models
plt.figure(figsize=(9, 7))

for name in models:
    precision, recall, _ = precision_recall_curve(
        y_test,
        scores[name]
    )
    ap = average_precision_score(y_test, scores[name])
    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves — Model Comparison")
plt.legend()
plt.tight_layout()
plt.show()


# 🔁 Cross-Validation

A single train/test split can be influenced by which observations happen to land in the test set. Five-fold stratified cross-validation gives a more stable estimate of model performance on the training portion.


In [ ]:
# STEP 33 — 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_rows = []

for name, estimator in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    cv_scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_rows.append({
        "Model": name,
        "CV ROC-AUC Mean": cv_scores.mean(),
        "CV ROC-AUC Std": cv_scores.std(),
        "Fold 1": cv_scores[0],
        "Fold 2": cv_scores[1],
        "Fold 3": cv_scores[2],
        "Fold 4": cv_scores[3],
        "Fold 5": cv_scores[4]
    })

cv_df = (
    pd.DataFrame(cv_rows)
    .sort_values("CV ROC-AUC Mean", ascending=False)
    .reset_index(drop=True)
)

display(cv_df.round(4))


In [ ]:
# STEP 34 — Cross-validation ROC-AUC comparison
plt.figure(figsize=(10, 6))

x = np.arange(len(cv_df))
means = cv_df["CV ROC-AUC Mean"]
stds = cv_df["CV ROC-AUC Std"]

plt.bar(x, means, yerr=stds, capsize=5)
plt.xticks(x, cv_df["Model"], rotation=30, ha="right")
plt.ylim(0, 1)
plt.ylabel("Mean ROC-AUC")
plt.title("5-Fold Cross-Validation ROC-AUC")
plt.tight_layout()
plt.show()


# 🌳 Random Forest Feature Importance

Feature importance shows which transformed input variables contributed most to the Random Forest's decisions. Higher importance means the feature was more useful to the fitted forest; it does **not** prove causation.


In [ ]:
# STEP 35 — Random Forest feature importance
rf_pipe = trained_models["Random Forest"]

feature_names = rf_pipe.named_steps["preprocessor"].get_feature_names_out()
importances = rf_pipe.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

plt.figure(figsize=(10, 7))
top_n = min(15, len(importance_df))

plt.barh(
    importance_df["Feature"].head(top_n)[::-1],
    importance_df["Importance"].head(top_n)[::-1]
)

plt.xlabel("Importance")
plt.title("Top Random Forest Feature Importances")
plt.tight_layout()
plt.show()

display(importance_df.head(15))


# 📈 Logistic Regression Coefficients

Because the numeric features are standardized inside the pipeline, the magnitude of a Logistic Regression coefficient can be compared across features. A positive coefficient pushes the model toward the diabetes class, while a negative coefficient pushes it toward the non-diabetes class.


In [ ]:
# STEP 36 — Logistic Regression coefficients
log_pipe = trained_models["Logistic Regression"]

log_feature_names = log_pipe.named_steps["preprocessor"].get_feature_names_out()
log_coefficients = log_pipe.named_steps["model"].coef_[0]

coef_df = (
    pd.DataFrame({
        "Feature": log_feature_names,
        "Coefficient": log_coefficients,
        "Absolute_Coefficient": np.abs(log_coefficients)
    })
    .sort_values("Absolute_Coefficient", ascending=False)
)

plt.figure(figsize=(10, 7))
top_n = min(15, len(coef_df))

plt.barh(
    coef_df["Feature"].head(top_n)[::-1],
    coef_df["Coefficient"].head(top_n)[::-1]
)

plt.axvline(0, linestyle="--")
plt.xlabel("Logistic Regression coefficient")
plt.title("Top Logistic Regression Coefficients")
plt.tight_layout()
plt.show()

display(coef_df.head(15))


# 📉 Linear Regression Diagnostic Graphs

Linear Regression is not the natural choice for a binary classification problem, but it is included to match the reference notebook. These diagnostics show why its continuous predictions should be interpreted differently from probability outputs of the classification models.


In [ ]:
# STEP 37 — Linear Regression predictions and residuals
linear_pipe = trained_models["Linear Regression"]

linear_pred = linear_pipe.predict(X_test)
linear_residuals = y_test.to_numpy() - linear_pred

plt.figure(figsize=(8, 5))
plt.scatter(linear_pred, linear_residuals, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted value")
plt.ylabel("Residual")
plt.title("Linear Regression — Residual Plot")
plt.tight_layout()
plt.show()


In [ ]:
# STEP 38 — Actual vs predicted values for Linear Regression
plt.figure(figsize=(8, 6))
plt.scatter(y_test, linear_pred, alpha=0.6)

minimum = min(y_test.min(), linear_pred.min())
maximum = max(y_test.max(), linear_pred.max())

plt.plot([minimum, maximum], [minimum, maximum], "--")
plt.xlabel("Actual Outcome")
plt.ylabel("Predicted value")
plt.title("Linear Regression — Actual vs Predicted")
plt.tight_layout()
plt.show()


In [ ]:
# STEP 39 — Prediction score distributions
plt.figure(figsize=(9, 6))

for outcome_value, label in [(0, "No Diabetes"), (1, "Diabetes")]:
    plt.hist(
        scores["Logistic Regression"][y_test.to_numpy() == outcome_value],
        bins=15,
        alpha=0.6,
        label=label
    )

plt.xlabel("Predicted probability of Diabetes")
plt.ylabel("Frequency")
plt.title("Logistic Regression — Predicted Probability Distribution")
plt.legend()
plt.tight_layout()
plt.show()


# 📚 Learning Curves

Learning curves show how model performance changes as more training examples are used. They can help identify whether a model is likely underfitting or overfitting.


In [ ]:
# STEP 40 — Learning curves for the best classification models
# Change this list if you want to inspect different models.
learning_curve_models = [
    "Logistic Regression",
    "Random Forest",
    "SVM"
]

for name in learning_curve_models:
    estimator = models[name]

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    train_sizes, train_scores, validation_scores = learning_curve(
        pipe,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        train_sizes=np.linspace(0.1, 1.0, 5),
        n_jobs=-1
    )

    train_mean = train_scores.mean(axis=1)
    validation_mean = validation_scores.mean(axis=1)

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, marker="o", label="Training ROC-AUC")
    plt.plot(train_sizes, validation_mean, marker="o", label="Validation ROC-AUC")
    plt.xlabel("Number of training examples")
    plt.ylabel("ROC-AUC")
    plt.ylim(0.4, 1.0)
    plt.title(f"Learning Curve — {name}")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# STEP 41 — Final ranking by important metrics
print("🏆 Best model by F1:")
print(
    results_df.sort_values("F1", ascending=False).iloc[0][
        ["Model", "F1", "Accuracy", "Precision", "Recall", "ROC_AUC", "PR_AUC"]
    ]
)

print("\n🏆 Best model by ROC-AUC:")
print(
    results_df.sort_values("ROC_AUC", ascending=False).iloc[0][
        ["Model", "F1", "Accuracy", "Precision", "Recall", "ROC_AUC", "PR_AUC"]
    ]
)

print("\n🏆 Best model by Recall:")
print(
    results_df.sort_values("Recall", ascending=False).iloc[0][
        ["Model", "F1", "Accuracy", "Precision", "Recall", "ROC_AUC", "PR_AUC"]
    ]
)

print("\nComplete test-set comparison:")
display(results_df.round(4))

print("\nCross-validation comparison:")
display(cv_df.round(4))


In [ ]:
# STEP 42 — Save cleaned data and model-comparison tables
df_model.to_csv("Diabetes_cleaned.csv", index=False)
results_df.to_csv("diabetes_model_comparison_results.csv", index=False)
cv_df.to_csv("diabetes_cross_validation_results.csv", index=False)
importance_df.to_csv("diabetes_random_forest_feature_importance.csv", index=False)
coef_df.to_csv("diabetes_logistic_regression_coefficients.csv", index=False)

print("All output files saved successfully.")


# ✅ Interpretation Checklist

When writing the project report, do not choose a model using accuracy alone.

For diabetes screening:
- **Recall** is important when missing a diabetic patient is costly.
- **Precision** matters when false alarms are costly.
- **F1** balances precision and recall.
- **ROC-AUC** evaluates ranking ability across classification thresholds.
- **PR-AUC** is especially useful when the positive class is less common.
- **Cross-validation ROC-AUC** helps assess stability across different training folds.
- Feature importance / coefficients describe model behavior, but they do not establish medical causation.

Finally, this dataset is suitable for an ML academic project, but a model trained on it should not be treated as a clinical diagnostic tool without external validation, calibration, appropriate clinical evaluation, and deployment safeguards.
